# Surrogates, uncertainty, and Bayesian optimization

<div style="background:linear-gradient(135deg,#eef3fb 0%,#f8fafc 100%);border:1px solid #d6e0ee;border-radius:12px;padding:18px 26px;color:#1b1b1b">
  <div style="font-size:12px;letter-spacing:.12em;text-transform:uppercase;color:#1f5fa8;font-weight:700">Hands-on workshop · ~60 min</div>
  <div style="font-size:16px;color:#3a3a3a;max-width:820px;margin-top:6px">From a CFD DOE table over Mach × angle of attack to <i>“which flight condition do we simulate next?”</i></div>
</div>

**You will learn to** build a `scaler → reducer → estimator` pipeline; read μ and σ as *maps* and spot a data gap from σ alone; split uncertainty into the part more CFD runs can fix and the part they can't; turn (μ, σ) into a decision.

Every figure is interactive (hover, zoom, sliders, dropdowns). Every code cell is editable and runs live — green boxes are exercises with a collapsed solution.

```{thebe-button} Launch live Python (Binder) — then run cells with the ▶ buttons
```

In [1]:
import uuid
import warnings
from dataclasses import dataclass
from enum import IntEnum, ReprEnum, StrEnum

import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display
from numpy.typing import NDArray
from plotly.offline import get_plotlyjs
from plotly.subplots import make_subplots
from scipy.linalg import cho_solve
from scipy.stats import norm
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

type FloatArray = NDArray[np.float64]
type Trace = go.Heatmap | go.Scatter


class Aero(float, ReprEnum):
    """Design box, atmosphere, synthetic wing, noise map and DOE hole.

    Attributes:
        MACH_MIN: Lower Mach bound.
        MACH_MAX: Upper Mach bound.
        AOA_MIN: Lower angle of attack, degrees.
        AOA_MAX: Upper angle of attack, degrees.
        SPEED_OF_SOUND: m/s.
        DENSITY: kg/m³.
        VISCOSITY: Pa·s.
        LIFT_SLOPE: Incompressible lift-curve slope per degree.
        ZERO_LIFT_AOA: Zero-lift angle, degrees.
        CL_MAX: Low-speed maximum lift coefficient.
        STALL_AOA: Low-speed stall angle, degrees.
        CD0: Parasite drag at low Mach.
        PARASITE_DROP: Fractional parasite-drag reduction across the Mach range.
        INDUCED_K: Induced-drag factor.
        MACH_DD: Drag-divergence Mach at zero lift.
        WAVE_DRAG: Lock's-law factor on (M − M_dd)⁴.
        NOISE_BASE: Noise std in clean attached flow.
        NOISE_STALL: Extra noise std deep in stall.
        NOISE_BUFFET: Extra noise std in transonic buffet.
        BUFFET_MACH: Mach where buffet noise switches on.
        GAP_MACH_LO: DOE hole, lower Mach edge.
        GAP_MACH_HI: DOE hole, upper Mach edge.
        GAP_AOA_LO: DOE hole, lower AoA edge.
        GAP_AOA_HI: DOE hole, upper AoA edge.
    """

    MACH_MIN = 0.2
    MACH_MAX = 0.95
    AOA_MIN = -2.0
    AOA_MAX = 16.0
    SPEED_OF_SOUND = 340.0
    DENSITY = 1.225
    VISCOSITY = 1.8e-5
    LIFT_SLOPE = 0.0986
    ZERO_LIFT_AOA = -2.0
    CL_MAX = 1.45
    STALL_AOA = 14.0
    CD0 = 0.02
    PARASITE_DROP = 0.5
    INDUCED_K = 0.0468
    MACH_DD = 0.72
    WAVE_DRAG = 40.0
    NOISE_BASE = 0.3
    NOISE_STALL = 1.5
    NOISE_BUFFET = 1.0
    BUFFET_MACH = 0.84
    GAP_MACH_LO = 0.55
    GAP_MACH_HI = 0.88
    GAP_AOA_LO = -0.5
    GAP_AOA_HI = 8.5


class Sizes(IntEnum):
    """Sample sizes and seeds.

    Attributes:
        GRID: Points per axis of the evaluation grid.
        POOL: Largest DOE drawn for the sliders.
        POOL_SEED: Seed of the nested DOE pool.
        DEMO: DOE size for the single-surrogate figures and exercises.
        BO_INIT: BO initial design size (first rows of the pool).
        BO_ITER: BO acquisition steps.
        BO_SEED: Observation-noise seed of the BO run.
        HEIGHT: Figure height in pixels.
    """

    GRID = 60
    POOL = 192
    POOL_SEED = 3
    DEMO = 96
    BO_INIT = 12
    BO_ITER = 16
    BO_SEED = 5
    HEIGHT = 480


class Acquisition(StrEnum):
    """Acquisition rules for run_bo.

    Attributes:
        EI: Expected improvement.
        PI: Probability of improvement.
        UCB: Upper confidence bound.
    """

    EI = "EI"
    PI = "PI"
    UCB = "UCB"


warnings.filterwarnings("ignore", category=ConvergenceWarning)
TRAIN_SIZES = (24, 48, 96, 192)
MACH_AXIS = np.linspace(Aero.MACH_MIN, Aero.MACH_MAX, Sizes.GRID)
AOA_AXIS = np.linspace(Aero.AOA_MIN, Aero.AOA_MAX, Sizes.GRID)
MACH_GRID, AOA_GRID = (g.ravel() for g in np.meshgrid(MACH_AXIS, AOA_AXIS))
PLOTLY_JS_EMITTED = [False]


def show(fig: go.Figure, title: str, subtitle: str = "") -> None:
    """Light theme, title, and render as HTML that draws once its element is in the page (Jupyter, Thebe, nbconvert, browser build).

    Args:
        fig: Figure to render.
        title: Headline.
        subtitle: Muted line under the headline.
    """
    sub = f"<br><sup><span style='color:#666'>{subtitle}</span></sup>" if subtitle else ""
    fig.update_layout(template="plotly_white", paper_bgcolor="white", plot_bgcolor="white", font=dict(size=13),
                      title=dict(text=f"<b>{title}</b>{sub}", x=0.0, xanchor="left", font=dict(size=17)))
    uid = uuid.uuid4().hex
    draw = (f'<script>(function(){{var fig={fig.to_json()};var tries=0;(function draw(){{var el=document.getElementById("{uid}");'
            f'if(!el||!window.Plotly){{if(tries++<400)setTimeout(draw,25);return;}}'
            f'Plotly.newPlot(el,fig.data,fig.layout,{{displaylogo:false,responsive:true}}).then(function(){{if(fig.frames)Plotly.addFrames(el,fig.frames);}});}})();}})();</script>')
    plotly_js = "" if PLOTLY_JS_EMITTED[0] else f"<script>{get_plotlyjs()}</script>"
    PLOTLY_JS_EMITTED[0] = True
    display(HTML(f'{plotly_js}<div id="{uid}"></div>{draw}'))


def animate(fig: go.Figure, frames: list[list[Trace]], labels: list[str], prefix: str = "", dropdown: bool = False) -> None:
    """Expose per-step trace lists through a slider or a dropdown.

    Args:
        fig: Figure whose base traces equal frames[0].
        frames: One trace list per step.
        labels: One label per step.
        prefix: Text before the active label (slider only).
        dropdown: Use a dropdown instead of a slider.
    """
    fig.frames = [go.Frame(data=data, name=label) for data, label in zip(frames, labels)]
    opts = dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))
    entries = [dict(method="animate", args=[[label], opts], label=label) for label in labels]
    if dropdown:
        fig.update_layout(updatemenus=[dict(buttons=entries, x=0, xanchor="left", y=1.2, yanchor="top")], margin=dict(t=120))
    else:
        fig.update_layout(sliders=[dict(steps=entries, currentvalue=dict(prefix=prefix), pad=dict(t=110), x=0.02, len=0.96)])

## 1. The problem: a DOE with a hole in it

Target: the wing's **lift-to-drag ratio** $L/D$ over Mach $M$ and angle of attack $\alpha$ — it has a real interior optimum (a best cruise condition) for section 5 to find. The synthetic "CFD" uses textbook pieces: Prandtl–Glauert lift slope, a stall that moves with Mach, induced + wave + separation drag.

* **Five columns, not two.** Run logs record $M$, $\alpha$ *and* $V_\infty$, $q_\infty$, $Re$ — three functions of Mach with magnitudes from $10^0$ to $10^7$.
* **The noise is a map.** Attached subsonic flow converges cleanly; stall and shock buffet ($M \gtrsim 0.84$) do not.
* **The DOE has a hole** — $M \in [0.55, 0.88]$, $\alpha \in [-0.5°, 8.5°]$ — right where cruise efficiency peaks.

In [2]:
def sigmoid(x: FloatArray) -> FloatArray:
    """Logistic function.

    Args:
        x: Inputs.

    Returns:
        1 / (1 + exp(-x)).
    """
    return 1.0 / (1.0 + np.exp(-x))


def stall_aoa(mach: FloatArray) -> FloatArray:
    """Stall angle, dropping in the transonic regime.

    Args:
        mach: Mach numbers.

    Returns:
        Stall angle in degrees.
    """
    return Aero.STALL_AOA - 7.0 * sigmoid((mach - 0.78) / 0.05)


def lift_to_drag(mach: FloatArray, aoa: FloatArray) -> FloatArray:
    """Noise-free L/D: compressible lift with smooth stall over parasite + induced + wave + separation drag.

    Args:
        mach: Mach numbers.
        aoa: Angles of attack in degrees.

    Returns:
        Lift-to-drag ratio.
    """
    post_stall = 2.0 * np.logaddexp(0.0, (aoa - stall_aoa(mach)) / 2.0)
    slope = Aero.LIFT_SLOPE / np.sqrt(1.0 - mach ** 2) * (1.0 - 0.45 * sigmoid((mach - 0.85) / 0.03))
    cl_max = Aero.CL_MAX * (1.0 - 0.5 * sigmoid((mach - 0.80) / 0.05))
    cl = cl_max * np.tanh(slope * (aoa - Aero.ZERO_LIFT_AOA) / cl_max) - 0.02 * post_stall ** 1.5
    parasite = Aero.CD0 * (1.0 - Aero.PARASITE_DROP * (mach - Aero.MACH_MIN) / (Aero.MACH_MAX - Aero.MACH_MIN))
    mach_dd = Aero.MACH_DD - 0.05 * np.clip(aoa, 0.0, Aero.AOA_MAX) / Aero.AOA_MAX
    cd = parasite + Aero.INDUCED_K * cl ** 2 + Aero.WAVE_DRAG * np.clip(mach - mach_dd, 0.0, None) ** 4 + 0.01 * post_stall ** 2
    return cl / cd


def noise_sigma(mach: FloatArray, aoa: FloatArray) -> FloatArray:
    """Noise std map: clean attached flow, noisy stall, noisy buffet.

    Args:
        mach: Mach numbers.
        aoa: Angles of attack in degrees.

    Returns:
        Noise std in L/D units.
    """
    stall = Aero.NOISE_STALL * sigmoid((aoa - stall_aoa(mach)) / 1.5)
    buffet = Aero.NOISE_BUFFET * sigmoid((mach - Aero.BUFFET_MACH) / 0.03)
    return Aero.NOISE_BASE + stall + buffet


def doe_table(mach: FloatArray, aoa: FloatArray) -> FloatArray:
    """The five columns a run log records: M, α, V∞, q∞, Re.

    Args:
        mach: Mach numbers.
        aoa: Angles of attack in degrees.

    Returns:
        Feature matrix (n, 5) in raw units.
    """
    velocity = mach * Aero.SPEED_OF_SOUND
    return np.column_stack([mach, aoa, velocity, 0.5 * Aero.DENSITY * velocity ** 2, Aero.DENSITY * velocity / Aero.VISCOSITY])


def in_gap(mach: FloatArray, aoa: FloatArray) -> NDArray[np.bool_]:
    """Mask of points inside the DOE hole.

    Args:
        mach: Mach numbers.
        aoa: Angles of attack in degrees.

    Returns:
        Boolean mask.
    """
    return (mach > Aero.GAP_MACH_LO) & (mach < Aero.GAP_MACH_HI) & (aoa > Aero.GAP_AOA_LO) & (aoa < Aero.GAP_AOA_HI)


def sample_doe(n: int, seed: int, gap: bool = True) -> tuple[FloatArray, FloatArray, FloatArray, FloatArray]:
    """The four box corners plus uniform random noisy CFD samples.

    Args:
        n: Samples kept.
        seed: Random seed for locations and noise.
        gap: Reject samples inside the DOE hole.

    Returns:
        Tuple of (mach, aoa, feature matrix X, noisy L/D y).
    """
    rng = np.random.default_rng(seed)
    mach = rng.uniform(Aero.MACH_MIN, Aero.MACH_MAX, size=4 * n)
    aoa = rng.uniform(Aero.AOA_MIN, Aero.AOA_MAX, size=4 * n)
    keep = ~in_gap(mach, aoa) if gap else np.ones(4 * n, dtype=bool)
    mach = np.append([Aero.MACH_MIN, Aero.MACH_MAX, Aero.MACH_MIN, Aero.MACH_MAX], mach[keep][:n - 4])
    aoa = np.append([Aero.AOA_MIN, Aero.AOA_MIN, Aero.AOA_MAX, Aero.AOA_MAX], aoa[keep][:n - 4])
    y = lift_to_drag(mach, aoa) + rng.normal(size=n) * noise_sigma(mach, aoa)
    return mach, aoa, doe_table(mach, aoa), y


X_GRID = doe_table(MACH_GRID, AOA_GRID)
Y_TRUE = lift_to_drag(MACH_GRID, AOA_GRID)
SIGMA_TRUE = noise_sigma(MACH_GRID, AOA_GRID)
I_OPT = int(np.argmax(Y_TRUE))
MACH_POOL, AOA_POOL, X_POOL, Y_POOL = sample_doe(Sizes.POOL, Sizes.POOL_SEED)
print(f"true optimum: L/D = {Y_TRUE[I_OPT]:.2f} at Mach {MACH_GRID[I_OPT]:.2f}, AoA {AOA_GRID[I_OPT]:.1f}° — inside the hole: {bool(in_gap(MACH_GRID[I_OPT], AOA_GRID[I_OPT]))}")

true optimum: L/D = 20.50 at Mach 0.76, AoA 1.7° — inside the hole: True


In [3]:
def fields_figure(titles: tuple[str, ...]) -> go.Figure:
    """Empty 1×k subplot figure with Mach / AoA axes.

    Args:
        titles: One subplot title per column.

    Returns:
        Figure ready for field and points traces.
    """
    fig = make_subplots(rows=1, cols=len(titles), subplot_titles=titles, horizontal_spacing=0.36 / len(titles))
    fig.update_xaxes(title_text="Mach")
    fig.update_yaxes(title_text="angle of attack [deg]", col=1)
    fig.update_layout(height=Sizes.HEIGHT, legend=dict(orientation="h", y=-0.15, x=0))
    return fig


def field(fig: go.Figure, z: FloatArray, col: int, colorscale: str, zrange: tuple[float, float] | None = None, name: str = "") -> go.Heatmap:
    """Design-space heatmap for subplot column col, colourbar parked right of that panel.

    Args:
        fig: Figure the trace will live in.
        z: Values on the flattened grid.
        col: 1-based subplot column.
        colorscale: Plotly colourscale.
        zrange: Fixed colour range.
        name: Colourbar title.

    Returns:
        Heatmap trace bound to that column's axes.
    """
    suffix = "" if col == 1 else str(col)
    zmin, zmax = zrange if zrange else (None, None)
    return go.Heatmap(x=MACH_AXIS, y=AOA_AXIS, z=z.reshape(Sizes.GRID, Sizes.GRID), colorscale=colorscale, zmin=zmin, zmax=zmax, zsmooth="best",
                      xaxis=f"x{suffix}", yaxis=f"y{suffix}", colorbar=dict(title=name, len=0.8, thickness=14, x=fig.layout[f"xaxis{suffix}"].domain[1] + 0.012),
                      hovertemplate="Mach %{x:.2f}<br>AoA %{y:.1f}°<br>%{z:.2f}<extra></extra>")


def points(mach: FloatArray, aoa: FloatArray, col: int, name: str = "CFD runs", color: str = "#d1495b", size: int = 7, symbol: str = "circle") -> go.Scatter:
    """Markers for subplot column col.

    Args:
        mach: Mach numbers.
        aoa: Angles of attack.
        col: 1-based subplot column.
        name: Legend name.
        color: Marker colour.
        size: Marker size.
        symbol: Marker symbol.

    Returns:
        Scatter trace bound to that column's axes.
    """
    suffix = "" if col == 1 else str(col)
    return go.Scatter(x=mach, y=aoa, mode="markers", name=name, xaxis=f"x{suffix}", yaxis=f"y{suffix}", showlegend=col == 1,
                      marker=dict(color=color, size=size, symbol=symbol, line=dict(color="white", width=1)), hovertemplate="Mach %{x:.2f}<br>AoA %{y:.1f}°<extra></extra>")


fig = fields_figure(("true L/D", "true noise σ"))
fig.add_traces([field(fig, Y_TRUE, 1, "Viridis", name="L/D"), points(MACH_POOL[:Sizes.DEMO], AOA_POOL[:Sizes.DEMO], 1, name=f"{Sizes.DEMO}-run DOE"),
                field(fig, SIGMA_TRUE, 2, "YlOrRd", name="σ"), points(MACH_POOL[:Sizes.DEMO], AOA_POOL[:Sizes.DEMO], 2)])
fig.add_shape(type="rect", x0=Aero.GAP_MACH_LO, x1=Aero.GAP_MACH_HI, y0=Aero.GAP_AOA_LO, y1=Aero.GAP_AOA_HI, line=dict(color="white", dash="dash", width=2), row=1, col=1)
show(fig, "The problem", "left: the field to learn, with the DOE hole outlined · right: how noisy the “CFD” is")

## 2. The pipeline: scaler → reducer → estimator

A surrogate is one `Pipeline` with three named steps, tuned together:

<svg viewBox="0 0 900 130" xmlns="http://www.w3.org/2000/svg" style="max-width:900px;background:#fff;font-family:sans-serif">
<defs><marker id="arr" markerWidth="10" markerHeight="10" refX="8" refY="5" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#4a4a4a"/></marker></defs>
<rect x="10" y="30" width="140" height="70" rx="8" fill="#f4f6fa" stroke="#9aa5b1"/><text x="80" y="58" text-anchor="middle" font-size="14" fill="#222">DOE table</text><text x="80" y="78" text-anchor="middle" font-size="11" fill="#666">M, α, V∞, q∞, Re</text><text x="80" y="92" text-anchor="middle" font-size="10" fill="#888">units 10⁰ … 10⁷</text>
<line x1="150" y1="65" x2="200" y2="65" stroke="#4a4a4a" stroke-width="2" marker-end="url(#arr)"/>
<rect x="205" y="30" width="150" height="70" rx="8" fill="#e8f0fb" stroke="#1f5fa8"/><text x="280" y="55" text-anchor="middle" font-size="14" fill="#1f5fa8">"scaler"</text><text x="280" y="76" text-anchor="middle" font-size="12" fill="#333">StandardScaler</text>
<line x1="355" y1="65" x2="405" y2="65" stroke="#4a4a4a" stroke-width="2" marker-end="url(#arr)"/>
<rect x="410" y="30" width="150" height="70" rx="8" fill="#e8f0fb" stroke="#1f5fa8"/><text x="485" y="55" text-anchor="middle" font-size="14" fill="#1f5fa8">"reducer"</text><text x="485" y="76" text-anchor="middle" font-size="12" fill="#333">PCA(2)</text>
<line x1="560" y1="65" x2="610" y2="65" stroke="#4a4a4a" stroke-width="2" marker-end="url(#arr)"/>
<rect x="615" y="30" width="150" height="70" rx="8" fill="#e8f0fb" stroke="#1f5fa8"/><text x="690" y="55" text-anchor="middle" font-size="14" fill="#1f5fa8">"estimator"</text><text x="690" y="76" text-anchor="middle" font-size="12" fill="#333">GP (RBF + White)</text>
<line x1="765" y1="65" x2="815" y2="65" stroke="#4a4a4a" stroke-width="2" marker-end="url(#arr)"/>
<text x="857" y="62" text-anchor="middle" font-size="14" fill="#222">μ(x), σ(x)</text><text x="857" y="82" text-anchor="middle" font-size="11" fill="#666">two fields</text></svg>

**Why scale first?** PCA chases variance. On the raw table the $Re$ column ($10^7$) *is* the variance, so PC1 would be "Reynolds number" and $\alpha$ would vanish. After scaling, the four speed-like columns collapse onto one direction and $\alpha$ gets the other.

In [4]:
def build_pipeline(n_components: int = 2, seed: int = 0) -> Pipeline:
    """Scaler → PCA → Gaussian process.

    Args:
        n_components: Principal components kept.
        seed: Seed for the GP optimizer restarts.

    Returns:
        Unfitted pipeline.
    """
    kernel = ConstantKernel(1.0, (1e-2, 1e3)) * RBF([0.5] * n_components, (1e-2, 5.0)) + WhiteKernel(0.1, (1e-5, 1e1))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=3, random_state=seed)
    return Pipeline([("scaler", StandardScaler()), ("reducer", PCA(n_components=n_components)), ("estimator", gp)])


def predict_parts(pipe: Pipeline, X: FloatArray) -> tuple[FloatArray, FloatArray, FloatArray]:
    """Mean, epistemic std and total std; total² = epistemic² + fitted noise variance.

    Args:
        pipe: Fitted pipeline.
        X: Feature matrix.

    Returns:
        Tuple of (mean, epistemic std, total std).
    """
    mean, std_total = pipe.predict(X, return_std=True)
    gp = pipe[-1]
    noise_var = float(gp.kernel_.k2.noise_level) * float(gp._y_train_std ** 2)
    return mean, np.sqrt(np.clip(std_total ** 2 - noise_var, 1e-12, None)), std_total


labels = [f"PC{i + 1}" for i in range(5)]
fig = go.Figure([go.Bar(x=labels, y=PCA().fit(X_POOL).explained_variance_ratio_, name="PCA on the raw table", marker_color="#b8c4d0"),
                 go.Bar(x=labels, y=PCA().fit(StandardScaler().fit_transform(X_POOL)).explained_variance_ratio_, name="PCA after StandardScaler", marker_color="#1f5fa8")])
fig.update_layout(height=360, yaxis_title="explained variance ratio", legend=dict(orientation="h", y=1.02, x=1, xanchor="right"))
show(fig, "Explained variance per component", "raw: PC1 is the Reynolds column · scaled: PC1 = speed, PC2 = angle of attack")

pipe_demo = build_pipeline().fit(X_POOL[:Sizes.DEMO], Y_POOL[:Sizes.DEMO])
latent = pipe_demo[:-1].transform(X_POOL[:Sizes.DEMO])
print("|corr(PC1, Mach)| =", round(abs(np.corrcoef(latent[:, 0], MACH_POOL[:Sizes.DEMO])[0, 1]), 3), "  |corr(PC2, AoA)| =", round(abs(np.corrcoef(latent[:, 1], AOA_POOL[:Sizes.DEMO])[0, 1]), 3))

|corr(PC1, Mach)| = 0.999   |corr(PC2, AoA)| = 0.997


<div style="border-left:5px solid #2a9d8f;background:#eef8f6;padding:12px 16px;border-radius:6px;margin:14px 0;color:#222;line-height:1.5"><b>🔧 Your turn 1 — walk a sample through the pipeline (5 min)</b><br><ol style='margin:6px 0 0 18px'><li>Which column has the largest <code>scale_</code>? Push three rows through <code>pipe_demo[:1]</code>, then <code>pipe_demo[:2]</code>.</li><li>Find the <code>get_params()</code> key for the number of PCA components.</li><li>Refit with <code>n_components=1</code> and compare the RMSE. What got thrown away?</li></ol></div>

In [5]:
print("stages:", list(pipe_demo.named_steps))
print("scaler scale_:", pipe_demo.named_steps["scaler"].scale_.round(2))
print("after scaler :", pipe_demo[:1].transform(X_POOL[:3]).round(2))
print("after reducer:", pipe_demo[:2].transform(X_POOL[:3]).round(2))
print("reducer params:", [k for k in pipe_demo.get_params() if k.startswith("reducer__")])
rmse = lambda pipe: np.sqrt(np.mean((pipe.predict(X_GRID) - Y_TRUE) ** 2))
print(f"RMSE vs truth, 2 components: {rmse(pipe_demo):.2f}")
# TODO: refit with reducer__n_components=1 and print its RMSE

stages: ['scaler', 'reducer', 'estimator']
scaler scale_: [2.20000000e-01 5.68000000e+00 7.53800000e+01 1.83950700e+04
 5.13032207e+06]
after scaler : [[-1.51 -1.73 -1.51 -1.14 -1.51]
 [ 1.87 -1.73  1.87  2.18  1.87]
 [-1.51  1.44 -1.51 -1.14 -1.51]]
after reducer: [[-2.91 -1.61]
 [ 3.82 -1.89]
 [-2.78  1.55]]
reducer params: ['reducer__copy', 'reducer__iterated_power', 'reducer__n_components', 'reducer__n_oversamples', 'reducer__power_iteration_normalizer', 'reducer__random_state', 'reducer__svd_solver', 'reducer__tol', 'reducer__whiten']
RMSE vs truth, 2 components: 1.28


<details style="margin:10px 0"><summary style="cursor:pointer;color:#1f5fa8;font-weight:600">Solution</summary><pre style="background:#f5f7f9;border:1px solid #dde3ea;border-radius:6px;padding:12px;font-size:12.5px"><code>pipe_1 = build_pipeline(n_components=1).fit(X_POOL[:Sizes.DEMO], Y_POOL[:Sizes.DEMO])
print(f&quot;RMSE vs truth, 1 component: {rmse(pipe_1):.2f}&quot;)   # ≈ 4.4 vs 0.8: AoA is gone, and L/D is mostly an AoA story</code></pre><p style='margin:8px 0 0 0'><code>set_params(step__param=…)</code> is how a hyperparameter search tunes a pipeline — same skeleton, different parameters.</p></details>

## 3. What a surrogate returns: `predict(X)` vs `predict(X, return_std=True)`

A fitted pipeline is called like any scikit-learn model. The one difference that matters for everything that follows is the `return_std` flag.
<div style="border-left:5px solid #2a9d8f;background:#eef8f6;padding:12px 16px;border-radius:6px;margin:14px 0;color:#222;line-height:1.5"><b>🔧 Your turn 2 — call predict both ways (5 min)</b><br><code>X_QUERY</code> holds five flight conditions, including one inside the DOE hole and one in buffet.<ol style='margin:6px 0 0 18px'><li>Run the cell: <code>predict(X_QUERY)</code> gives one array — the mean μ.</li><li>Change it to <code>predict(X_QUERY, return_std=True)</code>. You now get a <i>tuple</i> (μ, σ); unpack it and print both.</li><li>Which row has the largest σ, and why? Which has the smallest? Compare μ with the true L/D printed alongside.</li><li>Try <code>return_cov=True</code> instead — what shape comes back, and what are its diagonal entries?</li></ol></div>

In [6]:
QUERY_MACH = np.array([0.30, 0.60, 0.72, 0.90, 0.93])
QUERY_AOA = np.array([4.0, 10.0, 2.0, 12.0, 3.0])
X_QUERY = doe_table(QUERY_MACH, QUERY_AOA)
np.set_printoptions(precision=2, suppress=True)

print("X_QUERY (M, α, V∞, q∞, Re):\n", X_QUERY)
print("true L/D       :", lift_to_drag(QUERY_MACH, QUERY_AOA))
print("predict(X)     :", pipe_demo.predict(X_QUERY))
# TODO: mean, std = pipe_demo.predict(X_QUERY, return_std=True)

X_QUERY (M, α, V∞, q∞, Re):
 [[       0.3         4.        102.       6372.45  6941666.67]
 [       0.6        10.        204.      25489.8  13883333.33]
 [       0.72        2.        244.8     36705.31 16660000.  ]
 [       0.9        12.        306.      57352.05 20825000.  ]
 [       0.93        3.        316.2     61239.24 21519166.67]]
true L/D       : [16.87 15.2  20.22  1.81  4.92]
predict(X)     : [17.18 15.44 24.38  2.25  6.6 ]


<details style="margin:10px 0"><summary style="cursor:pointer;color:#1f5fa8;font-weight:600">Solution</summary><pre style="background:#f5f7f9;border:1px solid #dde3ea;border-radius:6px;padding:12px;font-size:12.5px"><code>mean, std = pipe_demo.predict(X_QUERY, return_std=True)
print(&quot;mean μ:&quot;, mean)
print(&quot;std  σ:&quot;, std)
print(&quot;in the DOE hole?&quot;, in_gap(QUERY_MACH, QUERY_AOA))
cov = pipe_demo.predict(X_QUERY, return_cov=True)[1]
print(&quot;cov shape:&quot;, cov.shape, &quot;| sqrt(diag) == σ:&quot;, np.allclose(np.sqrt(np.diag(cov)), std))</code></pre><p style='margin:8px 0 0 0'>Row 3 (Mach 0.72, 2°) sits in the hole: μ is a smooth guess and σ is the largest. Rows 1–2 sit in clean, well-sampled flow: smallest σ. Row 4 (stall) and row 5 (buffet) get the fitted noise floor plus whatever the data gap adds. <code>return_std</code> is the whole reason a surrogate is more than a curve fit.</p></details>

### Mean and σ as maps

Call `predict(X, return_std=True)` on a whole grid and the two vectors become two *fields*: $\mu(x)$, the best guess, and $\sigma(x)$, how far the truth could plausibly be from it. For a GP,

$$
\sigma^2(x_*) = \underbrace{k(x_*, x_*) - k_*^\top (K + \sigma_n^2 I)^{-1} k_*}_{\text{shrinks as data arrives}} + \underbrace{\sigma_n^2}_{\text{never shrinks}} .
$$

Drag the slider: the hole is filled with something smooth and *plausible* — plausible is not known — and σ darkens wherever runs land but stays bright in the hole, however many land elsewhere.

In [7]:
FITS = {n: build_pipeline().fit(X_POOL[:n], Y_POOL[:n]) for n in TRAIN_SIZES}
PARTS = {n: predict_parts(FITS[n], X_GRID) for n in TRAIN_SIZES}
STD_RANGE = (0.0, float(SIGMA_TRUE.max()))
LD_RANGE = (float(Y_TRUE.min()), float(Y_TRUE.max()))

fig = fields_figure(("surrogate mean μ", "surrogate std σ"))
frames = [[field(fig, PARTS[n][0], 1, "Viridis", LD_RANGE, "L/D"), points(MACH_POOL[:n], AOA_POOL[:n], 1),
           field(fig, PARTS[n][2], 2, "YlOrRd", STD_RANGE, "σ"), points(MACH_POOL[:n], AOA_POOL[:n], 2)] for n in TRAIN_SIZES]
fig.add_traces(frames[0])
animate(fig, frames, [f"n = {n}" for n in TRAIN_SIZES], prefix="CFD runs: ")
fig.update_layout(height=Sizes.HEIGHT + 110)
show(fig, "Mean and σ fields as the DOE grows", "σ is the map of where the model has never been")

Same surrogate read the aero way — $L/D$ versus $\alpha$ at fixed Mach, with the runs within ±0.05 Mach. Inner band: uncertainty from *missing data*; outer band: plus the fitted noise. Try $M = 0.75$ (through the hole) and $M = 0.90$ (buffet).

In [8]:
def slice_frame(mach: float) -> list[Trace]:
    """L/D vs AoA at fixed Mach for the demo surrogate: truth, bands, mean, nearby runs.

    Args:
        mach: Mach number of the slice.

    Returns:
        Trace list for one dropdown entry.
    """
    mean, std_epi, std_total = predict_parts(FITS[Sizes.DEMO], doe_table(np.full_like(AOA_AXIS, mach), AOA_AXIS))
    near = np.abs(MACH_POOL[:Sizes.DEMO] - mach) < 0.05
    closed = np.concatenate([AOA_AXIS, AOA_AXIS[::-1]])
    band = lambda s: np.concatenate([mean + 2 * s, (mean - 2 * s)[::-1]])
    return [go.Scatter(x=closed, y=band(std_total), fill="toself", fillcolor="rgba(237,139,0,0.25)", line_width=0, hoverinfo="skip", name="μ ± 2σ total"),
            go.Scatter(x=closed, y=band(std_epi), fill="toself", fillcolor="rgba(31,95,168,0.35)", line_width=0, hoverinfo="skip", name="μ ± 2σ epistemic"),
            go.Scatter(x=AOA_AXIS, y=lift_to_drag(np.full_like(AOA_AXIS, mach), AOA_AXIS), line=dict(color="#4a4a4a", dash="dash"), name="true L/D"),
            go.Scatter(x=AOA_AXIS, y=mean, line=dict(color="#1f5fa8", width=2.5), name="surrogate mean μ"),
            go.Scatter(x=AOA_POOL[:Sizes.DEMO][near], y=Y_POOL[:Sizes.DEMO][near], mode="markers", marker=dict(color="#d1495b", size=7), name="runs within ±0.05 Mach")]


MACH_SLICES = (0.3, 0.6, 0.75, 0.9)
fig = go.Figure(slice_frame(MACH_SLICES[0]))
animate(fig, [slice_frame(m) for m in MACH_SLICES], [f"Mach {m:.2f}" for m in MACH_SLICES], dropdown=True)
fig.update_layout(height=Sizes.HEIGHT, xaxis_title="angle of attack [deg]", yaxis_title="L/D", hovermode="x unified", legend=dict(orientation="h", y=-0.2))
show(fig, "L/D versus angle of attack at fixed Mach")

<div style="border-left:5px solid #2a9d8f;background:#eef8f6;padding:12px 16px;border-radius:6px;margin:14px 0;color:#222;line-height:1.5"><b>🔧 Your turn 3 — are the bands honest? (5 min)</b><br><code>coverage</code> draws fresh noisy runs and measures how often they fall inside μ ± 2σ (calibrated ≈ 0.95).<ol style='margin:6px 0 0 18px'><li>Whole design space: near 0.95?</li><li>Now with the <code>CLEAN</code> and <code>BUFFET</code> masks. What happens, and which assumption in <code>build_pipeline</code> is to blame?</li></ol></div>

In [9]:
MACH_TEST, AOA_TEST, X_TEST, Y_TEST = sample_doe(3000, seed=99, gap=False)
CLEAN = (AOA_TEST < 8) & (MACH_TEST < 0.8)
BUFFET = MACH_TEST >= Aero.BUFFET_MACH


def coverage(pipe: Pipeline, mask: NDArray[np.bool_] | None = None) -> float:
    """Fraction of fresh noisy runs inside μ ± 2σ.

    Args:
        pipe: Fitted pipeline.
        mask: Optional subset of the test runs.

    Returns:
        Empirical coverage.
    """
    keep = np.ones(Y_TEST.size, dtype=bool) if mask is None else mask
    mean, std = pipe.predict(X_TEST[keep], return_std=True)
    return float(np.mean(np.abs(Y_TEST[keep] - mean) <= 2 * std))


print(f"coverage over the whole design space: {coverage(FITS[Sizes.DEMO]):.3f}")
# TODO: coverage(FITS[Sizes.DEMO], CLEAN) and coverage(FITS[Sizes.DEMO], BUFFET)

coverage over the whole design space: 0.905


<details style="margin:10px 0"><summary style="cursor:pointer;color:#1f5fa8;font-weight:600">Solution</summary><pre style="background:#f5f7f9;border:1px solid #dde3ea;border-radius:6px;padding:12px;font-size:12.5px"><code>print(&quot;clean subsonic:&quot;, round(coverage(FITS[Sizes.DEMO], CLEAN), 3))   # ≈ 1.00 — bands too wide
print(&quot;buffet:        &quot;, round(coverage(FITS[Sizes.DEMO], BUFFET), 3))  # ≈ 0.7  — bands too narrow</code></pre><p style='margin:8px 0 0 0'>One <code>WhiteKernel</code> noise level: a homoscedastic assumption on heteroscedastic data. The headline 0.9 averages two wrong answers.</p></details><div style="border-left:5px solid #1f5fa8;background:#eef3fb;padding:12px 16px;border-radius:6px;margin:14px 0;color:#222;line-height:1.5"><b>In practice</b><br>Score coverage and CRPS alongside RMSE when selecting a surrogate, and use conformal recalibration to fix band width post hoc for any model family.</div>

## 4. Uncertainty decomposition

<svg viewBox="0 0 900 150" xmlns="http://www.w3.org/2000/svg" style="max-width:900px;background:#fff;font-family:sans-serif">
<rect x="330" y="10" width="240" height="40" rx="8" fill="#f4f6fa" stroke="#9aa5b1"/><text x="450" y="35" text-anchor="middle" font-size="14" fill="#222">total variance σ²(x) from return_std</text>
<line x1="400" y1="50" x2="230" y2="80" stroke="#4a4a4a" stroke-width="2"/><line x1="500" y1="50" x2="670" y2="80" stroke="#4a4a4a" stroke-width="2"/>
<rect x="60" y="80" width="340" height="60" rx="8" fill="#fff1dd" stroke="#ed8b00"/><text x="230" y="103" text-anchor="middle" font-size="14" fill="#8a4b00">aleatoric — noise in the CFD</text><text x="230" y="125" text-anchor="middle" font-size="11.5" fill="#8a4b00" font-weight="bold">more runs do NOT shrink it → model it, design around it</text>
<rect x="500" y="80" width="340" height="60" rx="8" fill="#e8f0fb" stroke="#1f5fa8"/><text x="670" y="103" text-anchor="middle" font-size="14" fill="#1f5fa8">epistemic — missing data</text><text x="670" y="125" text-anchor="middle" font-size="11.5" fill="#1f5fa8" font-weight="bold">more runs DO shrink it → sample where it is largest</text></svg>

$$
\underbrace{\mathrm{Var}[y \mid x]}_{\text{total}} = \underbrace{\mathbb{E}_\theta[\mathrm{Var}[y \mid x, \theta]]}_{\text{aleatoric}} + \underbrace{\mathrm{Var}_\theta[\mathbb{E}[y \mid x, \theta]]}_{\text{epistemic}}
$$

**Disentangling, two-stage:** a GP on $y$ gives $\mu$ and the epistemic variance; a second GP on the log squared **leave-one-out** residuals (closed form for a GP — in-sample residuals won't do, a GP's mean passes almost through its own points) gives a smooth noise map. Other routes to the same split: heteroscedastic GPs, BNN ensembles, conformal prediction.

Drag the slider: epistemic (left) collapses as runs arrive — except in the hole; aleatoric (middle) converges to the true noise map (right). With 24–48 runs the split is mostly noise; it becomes trustworthy around a hundred.

In [10]:
def noise_map(pipe: Pipeline, n: int) -> FloatArray:
    """Aleatoric std map from a second GP on log squared leave-one-out residuals, net of the epistemic part.

    Args:
        pipe: Pipeline fitted on the first n pool rows.
        n: Pool rows used.

    Returns:
        Aleatoric std on the grid.
    """
    gp = pipe[-1]
    precision = np.diag(cho_solve((gp.L_, True), np.eye(n)))
    scale = float(gp._y_train_std)
    residual = gp.alpha_.ravel() / precision * scale
    var_epi = (1.0 / precision - float(gp.kernel_.k2.noise_level)) * scale ** 2
    excess = np.maximum(residual ** 2 - var_epi, 0.25 * residual ** 2)
    return np.sqrt(np.exp(build_pipeline().fit(X_POOL[:n], np.log(excess + 1e-3)).predict(X_GRID) + 1.27))


fig = fields_figure(("epistemic std", "aleatoric std (learned)", "true noise σ"))
frames = [[field(fig, PARTS[n][1], 1, "YlOrRd", STD_RANGE, "σ"), points(MACH_POOL[:n], AOA_POOL[:n], 1),
           field(fig, noise_map(FITS[n], n), 2, "YlOrRd", STD_RANGE, "σ"), points(MACH_POOL[:n], AOA_POOL[:n], 2),
           field(fig, SIGMA_TRUE, 3, "YlOrRd", STD_RANGE, "σ"), points(MACH_POOL[:n], AOA_POOL[:n], 3)] for n in TRAIN_SIZES]
fig.add_traces(frames[0])
animate(fig, frames, [f"n = {n}" for n in TRAIN_SIZES], prefix="CFD runs: ")
fig.update_layout(height=Sizes.HEIGHT + 110)
show(fig, "Epistemic shrinks with data; aleatoric does not", "left: what more runs remove · middle: what they cannot · right: the target")

<div style="border-left:5px solid #e9a23b;background:#fff7e8;padding:12px 16px;border-radius:6px;margin:14px 0;color:#222;line-height:1.5"><b>💬 Discuss</b><br>You have 20 more CFD runs for this wing. Where do they go — for a <i>global surrogate</i>, and for <i>finding the best cruise condition</i>? Runs where the aleatoric map is bright mostly buy replicates.</div>

## 5. Bayesian optimization: uncertainty becomes a decision

An acquisition function scores every candidate from $(\mu, \sigma)$ alone; the next CFD run goes to its argmax.

<svg viewBox="0 0 900 140" xmlns="http://www.w3.org/2000/svg" style="max-width:900px;background:#fff;font-family:sans-serif">
<defs><marker id="arr2" markerWidth="10" markerHeight="10" refX="8" refY="5" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#4a4a4a"/></marker></defs>
<rect x="20" y="35" width="180" height="55" rx="8" fill="#e8f0fb" stroke="#1f5fa8"/><text x="110" y="67" text-anchor="middle" font-size="13.5" fill="#1f5fa8">1 · fit surrogate</text>
<line x1="200" y1="62" x2="250" y2="62" stroke="#4a4a4a" stroke-width="2" marker-end="url(#arr2)"/>
<rect x="255" y="35" width="180" height="55" rx="8" fill="#e8f0fb" stroke="#1f5fa8"/><text x="345" y="67" text-anchor="middle" font-size="13.5" fill="#1f5fa8">2 · score α(x) from μ, σ</text>
<line x1="435" y1="62" x2="485" y2="62" stroke="#4a4a4a" stroke-width="2" marker-end="url(#arr2)"/>
<rect x="490" y="35" width="180" height="55" rx="8" fill="#fff1dd" stroke="#ed8b00"/><text x="580" y="67" text-anchor="middle" font-size="13.5" fill="#8a4b00">3 · run CFD at argmax α</text>
<line x1="670" y1="62" x2="720" y2="62" stroke="#4a4a4a" stroke-width="2" marker-end="url(#arr2)"/>
<rect x="725" y="35" width="155" height="55" rx="8" fill="#f4f6fa" stroke="#9aa5b1"/><text x="802" y="67" text-anchor="middle" font-size="13.5" fill="#222">4 · append, repeat</text>
<path d="M 802 90 L 802 118 L 110 118 L 110 95" fill="none" stroke="#4a4a4a" stroke-width="2" marker-end="url(#arr2)"/></svg>

Maximizing $L/D$, with $f^+$ the best so far and $z = (\mu - f^+)/\sigma$:

| rule | formula | character |
|---|---|---|
| PI | $\Phi(z)$ | greedy |
| **EI** | $\sigma\,[\,z\,\Phi(z) + \phi(z)\,]$ | weighs *how much* improvement |
| UCB | $\mu + \kappa\,\sigma$ | $\kappa$ dials exploration |

Every rule multiplies or adds σ: **no uncertainty, no exploration.** The σ that belongs here is the *epistemic* one — it rewards going where the model hasn't been (the hole), not where the CFD is noisy — with $f^+ = \max_i \mu(x_i)$ so a lucky noisy draw can't set the bar. The loop starts from the 12-run DOE with the hole, so it has never seen the optimum.

In [11]:
@dataclass(frozen=True)
class BOStep:
    """One iteration of the optimizer.

    Attributes:
        mach: Mach numbers run so far.
        aoa: Angles of attack run so far.
        mean: Surrogate mean on the grid.
        acquisition: Acquisition scores on the grid.
        next_index: Grid index of the next run.
        best_index: Grid index of the recommended condition (highest μ).
    """

    mach: FloatArray
    aoa: FloatArray
    mean: FloatArray
    acquisition: FloatArray
    next_index: int
    best_index: int


def acquisition(rule: Acquisition, mean: FloatArray, std: FloatArray, incumbent: float, kappa: float) -> FloatArray:
    """Score candidates under one rule (maximization).

    Args:
        rule: Acquisition rule.
        mean: Predictive mean.
        std: Predictive epistemic std.
        incumbent: Best value so far.
        kappa: Exploration weight (UCB only).

    Returns:
        Score per candidate.
    """
    z = (mean - incumbent) / np.maximum(std, 1e-9)
    if rule == Acquisition.PI:
        return norm.cdf(z)
    if rule == Acquisition.UCB:
        return mean + kappa * std
    return std * (z * norm.cdf(z) + norm.pdf(z))


def run_bo(rule: Acquisition = Acquisition.EI, kappa: float = 2.0, seed: int = Sizes.BO_SEED) -> list[BOStep]:
    """Surrogate-driven search for the best cruise condition from the 12-run DOE.

    Args:
        rule: Acquisition rule.
        kappa: Exploration weight (UCB only).
        seed: Observation-noise seed.

    Returns:
        One BOStep per iteration.
    """
    rng = np.random.default_rng(seed)
    mach, aoa, y = MACH_POOL[:Sizes.BO_INIT], AOA_POOL[:Sizes.BO_INIT], Y_POOL[:Sizes.BO_INIT]
    steps = []
    for _ in range(Sizes.BO_ITER):
        pipe = build_pipeline(seed=seed).fit(doe_table(mach, aoa), y)
        mean, std_epi, _ = predict_parts(pipe, X_GRID)
        acq = acquisition(rule, mean, std_epi, float(pipe.predict(doe_table(mach, aoa)).max()), kappa)
        nxt = int(np.argmax(acq))
        steps.append(BOStep(mach, aoa, mean, acq, nxt, int(np.argmax(mean))))
        mach, aoa = np.append(mach, MACH_GRID[nxt]), np.append(aoa, AOA_GRID[nxt])
        y = np.append(y, Y_TRUE[nxt] + rng.normal() * SIGMA_TRUE[nxt])
    return steps


def regret(steps: list[BOStep]) -> FloatArray:
    """True L/D at the optimum minus true L/D at the recommended condition, per iteration.

    Args:
        steps: BO iterations.

    Returns:
        Regret per iteration.
    """
    return np.array([Y_TRUE[I_OPT] - Y_TRUE[s.best_index] for s in steps])


BO_STEPS = run_bo()
fig = fields_figure(("surrogate mean μ and CFD runs", "expected improvement"))
star = dict(name="next CFD run", color="#e76f51", size=16, symbol="star")
frames = [[field(fig, s.mean, 1, "Viridis", LD_RANGE, "μ"), points(s.mach, s.aoa, 1), points(MACH_GRID[[s.next_index]], AOA_GRID[[s.next_index]], 1, **star),
           field(fig, s.acquisition, 2, "Teal", name="EI"), points(MACH_GRID[[s.next_index]], AOA_GRID[[s.next_index]], 2, **star)] for s in BO_STEPS]
fig.add_traces(frames[0])
animate(fig, frames, [str(i) for i in range(Sizes.BO_ITER)], prefix="BO iteration: ")
fig.add_trace(points(MACH_GRID[[I_OPT]], AOA_GRID[[I_OPT]], 1, name="true optimum", color="#1b1b1b", size=13, symbol="x"))
fig.update_layout(height=Sizes.HEIGHT + 110)
show(fig, "EI picks the next CFD run from μ and σ", f"it heads into the hole first — regret {regret(BO_STEPS)[-1]:.2f} L/D after {Sizes.BO_ITER} runs")

Same 12-run start and budget, EI versus random sampling, eight noise seeds. **Regret** = true $L/D$ at the optimum minus true $L/D$ at the recommended condition. Random gets lucky now and then; it cannot aim.

In [12]:
def regret_random(seed: int) -> FloatArray:
    """Regret per iteration when the extra runs are placed uniformly at random.

    Args:
        seed: Random seed for the extra runs and their noise.

    Returns:
        Regret per iteration.
    """
    mach, aoa, _, y = sample_doe(Sizes.BO_INIT + Sizes.BO_ITER, seed=1000 + seed, gap=False)
    mach[:Sizes.BO_INIT], aoa[:Sizes.BO_INIT], y[:Sizes.BO_INIT] = MACH_POOL[:Sizes.BO_INIT], AOA_POOL[:Sizes.BO_INIT], Y_POOL[:Sizes.BO_INIT]
    fits = [build_pipeline(seed=seed).fit(doe_table(mach[:k], aoa[:k]), y[:k]) for k in range(Sizes.BO_INIT, mach.size)]
    return np.array([Y_TRUE[I_OPT] - Y_TRUE[int(np.argmax(pipe.predict(X_GRID)))] for pipe in fits])


SEEDS = range(Sizes.BO_SEED, Sizes.BO_SEED + 8)
fig = go.Figure()
iters = np.arange(Sizes.BO_ITER)
for name, curves, color, fill in (("EI-driven BO", [regret(run_bo(seed=s)) for s in SEEDS], "#1f5fa8", "rgba(31,95,168,0.3)"),
                                   ("random sampling", [regret_random(s) for s in SEEDS], "#8a8a8a", "rgba(138,138,138,0.25)")):
    lo, med, hi = np.percentile(np.stack(curves), [25, 50, 75], axis=0)
    fig.add_scatter(x=np.concatenate([iters, iters[::-1]]), y=np.concatenate([hi, lo[::-1]]), fill="toself", fillcolor=fill, line_width=0, hoverinfo="skip", showlegend=False)
    fig.add_scatter(x=iters, y=med, line=dict(color=color, width=2.5), name=name)
fig.update_layout(height=360, xaxis_title="iteration", yaxis_title="regret [L/D]", hovermode="x unified", legend=dict(orientation="h", y=1.02, x=1, xanchor="right"))
show(fig, "Regret vs. iteration", "median and inter-quartile range over seeds")

<div style="border-left:5px solid #2a9d8f;background:#eef8f6;padding:12px 16px;border-radius:6px;margin:14px 0;color:#222;line-height:1.5"><b>🔧 Your turn 4 — change the rule (5 min)</b><br><ol style='margin:6px 0 0 18px'><li>Run <code>run_bo(Acquisition.PI)</code> and UCB with <code>kappa=0.5</code> and <code>kappa=4</code>; print each final <code>regret</code>.</li><li>Print the last four runs of each. Who locks onto one condition and re-runs it? Who keeps probing?</li><li><i>Discuss:</i> what changes if the acquisition saw the <i>aleatoric</i> map from section 4 instead?</li></ol></div>

In [13]:
def last_runs(steps: list[BOStep], k: int = 4) -> list[tuple[float, float]]:
    """Mach/AoA of the last k CFD runs chosen.

    Args:
        steps: BO iterations.
        k: Number of final runs to report.

    Returns:
        List of (Mach, AoA) pairs.
    """
    return [(round(float(MACH_GRID[s.next_index]), 2), round(float(AOA_GRID[s.next_index]), 1)) for s in steps[-k:]]


print("EI  regret", f"{regret(BO_STEPS)[-1]:.2f}", last_runs(BO_STEPS))
# TODO: same for PI, UCB kappa=0.5, UCB kappa=4

EI  regret 1.04 [(0.67, 3.2), (0.67, 3.2), (0.67, 3.2), (0.67, 2.9)]


<details style="margin:10px 0"><summary style="cursor:pointer;color:#1f5fa8;font-weight:600">Solution</summary><pre style="background:#f5f7f9;border:1px solid #dde3ea;border-radius:6px;padding:12px;font-size:12.5px"><code>for label, steps in ((&quot;PI &quot;, run_bo(Acquisition.PI)), (&quot;UCB 0.5&quot;, run_bo(Acquisition.UCB, kappa=0.5)), (&quot;UCB 4  &quot;, run_bo(Acquisition.UCB, kappa=4.0))):
    print(label, f&quot;regret {regret(steps)[-1]:.2f}&quot;, last_runs(steps))</code></pre><p style='margin:8px 0 0 0'>The σ map is so lopsided that every rule heads into the hole; they differ in the endgame — PI and greedy UCB lock onto one condition and re-run it, κ = 4 keeps probing, EI sits between. Re-running a known point is the sign to stop — that is what a confidence-bound stopping rule detects. With the aleatoric map in the loop, stall and buffet would look permanently interesting and the budget would go to replicates.</p></details>

## 6. Wrap-up

| here | in a production surrogate workflow |
|---|---|
| hand-built `StandardScaler → PCA → GP` | pipeline assembled and tuned by a hyperparameter search |
| `predict(X, return_std=True)` | the two outputs every surrogate should expose |
| exercise 3's broken coverage | coverage / CRPS in model selection; conformal recalibration |
| two-stage noise map | heteroscedastic GPs, BNN ensembles, conformalised tree ensembles |
| `run_bo` with EI on epistemic σ | Bayesian optimization with a confidence-bound stopping rule |

1. **μ is a guess; σ is how much to trust it** — report both, and check σ is honest.
2. **Epistemic uncertainty is a to-do list; aleatoric uncertainty is a fact of life.**
3. **Optimization without uncertainty is hill-climbing.**
<div style="border-left:5px solid #e9a23b;background:#fff7e8;padding:12px 16px;border-radius:6px;margin:14px 0;color:#222;line-height:1.5"><b>💬 Closing</b><br>Pick a case your team runs. Which DOE columns would a reducer collapse? Where does its noise come from — and is any of it solver noise you could reduce?</div>